In [43]:
# Actualizar repositorios e instalar Java
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq -y

# Descargar Spark (usando versión estable del archivo)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar xf spark-3.5.7-bin-hadoop3.tgz

# Instalar findspark
!pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [44]:
!pip install opendatasets
import opendatasets as od

In [28]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [45]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"

In [46]:
!ls

2019-Nov.csv  spark-3.5.7-bin-hadoop3
2019-Oct.csv  spark-3.5.7-bin-hadoop3.tgz


In [47]:
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) 
spark

In [ ]:
# from google.colab import drive  # type: ignore
# drive.mount('/content/drive')

In [50]:
# dataset_link="https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store"
dataset_link="https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store?select=2019-Nov.csv"
od.download(dataset_link)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Your Kaggle Key:Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store


100%|██████████| 4.29G/4.29G [00:58<00:00, 78.4MB/s]



In [51]:
import os
os.chdir("ecommerce-behavior-data-from-multi-category-store")
os.listdir()

['2019-Nov.csv', '2019-Oct.csv']

# 1. Análisis profundo de los datos


## 1.1 Estructura de los datos

In [52]:
# Lectura con esquema explícito (evita inferSchema que escanea 2 veces)
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

df = spark.read.csv('ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv', header=True, schema=schema)
df.show(5)
print(f"Columnas: {df.columns}")

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [53]:
# Esquema del DataFrame
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [13]:
df.describe()

summary,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
count,42448764,42448764,42448764,28933155,36335756,42448764,42448764,42448762
mean,NULL,1.0549932375842676E7,2.057404237936260...,NULL,NaN,290.3236606848809,5.335371475081686E8,NULL
stddev,NULL,1.1881906970608136E7,1.843926466140411...,NULL,NaN,358.2691553394021,1.852373817465431E7,NULL
min,cart,1000978,2053013552226107603,accessories.bag,a-case,0.0,33869381,00000042-3e3f-42f...
max,view,60500010,2175419595093967522,stationery.cartrige,zyxel,2574.07,566280860,fffffc65-7ce9-435...


__Valores únicos por columna__

In [15]:
from pyspark.sql.functions import countDistinct

df.agg(
    countDistinct("event_type").alias("valores_unicos_event_type"),
    countDistinct("product_id").alias("valores_unicos_product_id"),
    countDistinct("category_id").alias("valores_unicos_category_id"),
    countDistinct("category_code").alias("valores_unicos_category_code"),
    countDistinct("brand").alias("valores_unicos_brand"),
    countDistinct("price").alias("valores_unicos_price"),
    countDistinct("user_id").alias("valores_unicos_user_id"),
).show()

+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+
|valores_unicos_event_type|valores_unicos_product_id|valores_unicos_category_id|valores_unicos_category_code|valores_unicos_brand|valores_unicos_price|valores_unicos_user_id|
+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+
|                        3|                   166794|                       624|                         126|                3445|               65298|               3022290|
+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+



__Registros duplicados__

In [55]:
# Duplicados en una sola pasada (reutiliza para limpieza)
df_sin_dup = df.dropDuplicates()
duplicados = df.count() - df_sin_dup.count()
print(f"Duplicados: {duplicados}")

Duplicados: 30220


In [56]:
# Reutiliza df_sin_dup de la celda anterior + limpieza de nulos en una sola operación
from pyspark.sql.functions import when, col, coalesce, lit

df_clean = df_sin_dup.withColumn(
    "category_code_clean", coalesce(col("category_code"), lit("unknown"))
).withColumn(
    "brand_clean", coalesce(col("brand"), lit("no_brand"))
)

**Reemplazo de valores nulos en columnas category_code y brand** (Se les reasigna un nuevo valor a los registros con valores nulos)

In [57]:
# Cache estratégico después de limpieza
df_clean.cache()
print(f"Registros en df_clean: {df_clean.count()}")

Registros en df_clean: 42418544


In [59]:
df_clean.show(10)

+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+--------------------+-----------+
|         event_time|event_type|product_id|        category_id|       category_code|  brand| price|  user_id|        user_session| category_code_clean|brand_clean|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+--------------------+-----------+
|2019-10-01 00:06:50|      view|   3700766|2053013565983425517|appliances.enviro...|samsung|128.45|555448298|a4378fe7-5e6a-4b9...|appliances.enviro...|    samsung|
|2019-10-01 00:16:16|      view|  28401058|2053013566209917945|     accessories.bag|  karya|100.39|547093079|6619868d-16c8-401...|     accessories.bag|      karya|
|2019-10-01 02:17:52|      view|   1004659|2053013555631882655|electronics.smart...|samsung|787.18|519447668|264c64cd-e48d-44d...|electronics.smart...|    samsung|
|2019-10-01 02:1

## 1.2 Análisis temporal

In [35]:
from pyspark.sql.functions import min, max

df_clean.select(
    min("event_time").alias("fecha_minima"),
    max("event_time").alias("fecha_maxima")
).show()

+-------------------+-------------------+
|       fecha_minima|       fecha_maxima|
+-------------------+-------------------+
|2019-10-01 00:00:00|2019-10-31 23:59:59|
+-------------------+-------------------+



In [ ]:
from pyspark.sql.functions import to_date, hour, dayofweek

# Análisis temporal en una sola pasada con cache compartido
analisis_temporal = df_clean.select(
    dayofweek(col("event_time")).alias("dia_semana"),
    hour(col("event_time")).alias("hora_dia"),
    to_date(col("event_time")).alias("fecha")
).cache()

print("Eventos por día de la semana:")
analisis_temporal.groupBy("dia_semana").count().orderBy("dia_semana").show()

+----------+-------+
|dia_semana|  count|
+----------+-------+
|         1|5851611|
|         2|5317662|
|         3|6797348|
|         4|6648353|
|         5|6376063|
|         6|5824835|
|         7|5602672|
+----------+-------+



In [ ]:
# Reutiliza cache de analisis_temporal
analisis_temporal.groupBy("hora_dia").count().orderBy(col("count").desc()).show(10)

+--------+-------+
|hora_dia|  count|
+--------+-------+
|      16|3053226|
|      15|2979082|
|      17|2732443|
|      14|2676546|
|       8|2387991|
|      13|2353321|
|       9|2349336|
|       7|2333480|
|      10|2295245|
|       6|2266954|
+--------+-------+
only showing top 10 rows



In [ ]:
# Reutiliza cache
eventos_por_dia = analisis_temporal.groupBy("fecha").count()
eventos_por_dia.describe("count").show()

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|                31|
|   mean|1368340.1290322582|
| stddev|120140.19894682542|
|    min|           1126624|
|    max|           1638290|
+-------+------------------+



## 1.3 Análisis de comportamiento de usuarios

**Distribución de tipos de eventos**

In [ ]:
# Distribución de eventos con cache para reutilizar en tasas de conversión
from pyspark.sql.functions import sum as spark_sum

distribucion_eventos = df_clean.groupBy("event_type").count().cache()
total_eventos = distribucion_eventos.agg(spark_sum("count")).collect()[0][0]

distribucion_eventos.withColumn("porcentaje", col("count") / total_eventos * 100) \
    .orderBy(col("count").desc()).show()

+----------+--------+------------------+
|event_type|   count|        porcentaje|
+----------+--------+------------------+
|      view|40777328|  96.1308997310233|
|      cart|  898443|2.1180429955351605|
|  purchase|  742773|1.7510572734415402|
+----------+--------+------------------+



In [ ]:
# Reutiliza distribucion_eventos cacheado (evita 4 count() separados)
conteos = {row["event_type"]: row["count"] for row in distribucion_eventos.collect()}
views = conteos.get("view", 0)
carts = conteos.get("cart", 0)
purchases = conteos.get("purchase", 0)

print("Tasas de conversión globales:")
print(f"Views: {views:,} ({views/total_eventos*100:.2f}%)")
print(f"Carts: {carts:,} ({carts/total_eventos*100:.2f}%)")
print(f"Purchase: {purchases:,} ({purchases/total_eventos*100:.2f}%)")
print(f"\nTasa cart/view: {carts/views*100:.2f}%")
print(f"Tasa purchase/view: {purchases/views*100:.2f}%")
print(f"Tasa purchase/cart: {purchases/carts*100:.2f}%")

Tasas de conversión globales: 
Views: 40,777,328 (96.13%)
Carts: 898,443 (2.12%)
Purchase: 742,773 (1.75%)

Tasa cart/view: 2.20%
Tasa purchase/view: 1.82%
Tasa purchase/cart: 82.67%


**Métricas de actividad por usuario**

In [ ]:
# Reutiliza eventos_por_tipo_usuario 
# Ejecutar primero eventos por tipo (celda de abajo)
eventos_por_usuario = eventos_por_tipo_usuario.withColumn(
    "total_eventos", col("cart") + col("purchase") + col("view")
)
eventos_por_usuario.describe("total_eventos").show()

+-------+-----------------+
|summary|    total_eventos|
+-------+-----------------+
|  count|          3022290|
|   mean| 14.0352328863213|
| stddev|32.75705325743598|
|    min|                1|
|    max|             7436|
+-------+-----------------+



In [58]:
# Distribución en percentiles
eventos_por_usuario.selectExpr(
    "percentile_approx(total_eventos, 0.25) as p25",
    "percentile_approx(total_eventos, 0.50) as p50_median",
    "percentile_approx(total_eventos, 0.75) as p75",
    "percentile_approx(total_eventos, 0.90) as p90",
    "percentile_approx(total_eventos, 0.95) as p95",
    "percentile_approx(total_eventos, 0.99) as p99"
).show()

+---+----------+---+---+---+---+
|p25|p50_median|p75|p90|p95|p99|
+---+----------+---+---+---+---+
|  2|         4| 13| 34| 57|140|
+---+----------+---+---+---+---+



**Eventos por tipo**

In [ ]:
# Pivot directo (evita doble groupBy) + cache para múltiples usos
eventos_por_tipo_usuario = df_clean.groupBy("user_id").pivot("event_type").count().fillna(0).cache()
eventos_por_tipo_usuario.show(5)

+---------+----+--------+----+
|  user_id|cart|purchase|view|
+---------+----+--------+----+
|514700043|   0|       0|   4|
|523367237|   1|       1|  53|
|519705698|   0|       0|  89|
|519798460|   1|       2| 102|
|547283407|   3|       3|  13|
+---------+----+--------+----+
only showing top 5 rows



In [ ]:
# Una sola llamada describe para los 3 tipos
eventos_por_tipo_usuario.select("view", "cart", "purchase").describe().show()

+-------+------------------+
|summary|              view|
+-------+------------------+
|  count|           3022290|
|   mean|13.492195652965135|
| stddev|31.855348368395525|
|    min|                 0|
|    max|              7436|
+-------+------------------+

+-------+------------------+
|summary|              cart|
+-------+------------------+
|  count|           3022290|
|   mean|0.2972722670557756|
| stddev|1.5011636291005397|
|    min|                 0|
|    max|               356|
+-------+------------------+

+-------+------------------+
|summary|          purchase|
+-------+------------------+
|  count|           3022290|
|   mean|0.2457649663003881|
| stddev|1.4093212679874512|
|    min|                 0|
|    max|               321|
+-------+------------------+



**Compradores vs navegadores**

In [ ]:
# Reutiliza eventos_por_tipo_usuario cacheado
total_usuarios = eventos_por_tipo_usuario.count()
compradores = eventos_por_tipo_usuario.filter(col("purchase") > 0).count()
navegadores = total_usuarios - compradores

print(f"Total usuarios: {total_usuarios:,}")
print(f"Compradores: {compradores:,} ({compradores/total_usuarios*100:.2f}%)")
print(f"Navegadores: {navegadores:,} ({navegadores/total_usuarios*100:.2f}%)")
print(f"Ratio: {compradores/navegadores:.2f}")

Total de usuarios: 3,022,290
Compradores: 347,118 (11.49%)
Solo navegadores: 2,675,172 (88.51%)
Ratio compradores/navegadores: 0.13


**Análisis de compras por usuario**

In [62]:
from pyspark.sql.functions import count, avg

compras_por_usuario = df_clean.filter(col("event_type") == "purchase") \
    .groupBy("user_id").agg(
        count("*").alias("num_compras"),
        spark_sum("price").alias("total_gastado"),
        avg("price").alias("gasto_promedio"),
        min("price").alias("gasto_minimo"),
        max("price").alias("gasto_maximo")
    )
    
print("Estadísticas de compras por usuario:")
compras_por_usuario.describe().show()

Estadísticas de compras por usuario:
+-------+--------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|             user_id|       num_compras|     total_gastado|   gasto_promedio|     gasto_minimo|      gasto_maximo|
+-------+--------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              347118|            347118|            347118|           347118|           347118|            347118|
|   mean| 5.359970205283938E8|2.1398285309318443| 662.4064803035074|278.0298408672455|241.1867638670545|320.26698992851243|
| stddev|1.8498600835749354E7|3.6387369214359544|2074.2141910618348|311.2153181859061|297.9926696586466| 360.8857670524742|
|    min|           264649825|                 1|              0.88|             0.88|             0.77|              0.88|
|    max|           566278294|               321|265569.51999999984|          2574.04|         

## Tasa de conversión por usuario

In [30]:
# Calculo de tasa de conversión por usuario
conversion_por_usuario = eventos_por_tipo_usuario.withColumn(
    "tasa_de_conversion",
    when(col("view") > 0, col("purchase") / col("view") * 100).otherwise(0)
).withColumn(
    "cart_rate",
    when(col("view") > 0, col("cart") / col("view") * 100).otherwise(0)
)

print("Distribución de tasas de conversión por usuario:")
conversion_por_usuario.select("tasa_de_conversion", "cart_rate").describe().show()

Distribución de tasas de conversión por usuario:
+-------+------------------+------------------+
|summary|tasa_de_conversion|         cart_rate|
+-------+------------------+------------------+
|  count|           3022290|           3022290|
|   mean| 2.003747909988533| 2.527292187227874|
| stddev| 8.888059905115897|13.473155834735465|
|    min|               0.0|               0.0|
|    max|             200.0|            2800.0|
+-------+------------------+------------------+



## 1.4 Análisis de productos y categorías

**Catálogo de productos**

In [34]:
# Total de productos únicos
total_productos = df_clean.select("product_id").distinct().count()
print(f"Total de productos únicos: {total_productos:,}")

Total de productos únicos: 166,794


**Tasa de conversión por producto**

In [ ]:
# Conversión por producto en una sola pasada (evita join)
conversion_producto = df_clean.groupBy("product_id").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("num_views"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("num_purchases")
).withColumn(
    "tasa_conversion_producto",
    when(col("num_views") > 0, col("num_purchases") / col("num_views") * 100).otherwise(0)
).cache()

print("Distribución de tasas de conversión por producto:")
conversion_producto.describe("tasa_conversion_producto").show()

Distribución de tasas de conversión por producto:
+-------+------------------------+
|summary|tasa_conversion_producto|
+-------+------------------------+
|  count|                  166794|
|   mean|      0.5676354841306908|
| stddev|       2.236626626845516|
|    min|                     0.0|
|    max|                   100.0|
+-------+------------------------+



**Categorías más populares**

In [ ]:
# Categorías populares en una sola pasada
stats_categoria = df_clean.groupBy("category_code_clean").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("views"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
    spark_sum(when(col("event_type") == "purchase", col("price")).otherwise(0)).alias("ganancias")
).cache()

print("Top 10 categorías más vistas:")
stats_categoria.orderBy(col("views").desc()).show(10, truncate=False)
print("Top 10 categorías con más compras:")
stats_categoria.orderBy(col("purchases").desc()).show(10, truncate=False)
print("Top categorías por ganancias:")
stats_categoria.orderBy(col("ganancias").desc()).show(10, truncate=False)

Top 10 categorías más vistas:
+--------------------------------+--------+
|category_code_clean             |count   |
+--------------------------------+--------+
|unknown                         |13235871|
|electronics.smartphone          |10618648|
|electronics.clocks              |1272733 |
|computers.notebook              |1106321 |
|electronics.video.tv            |1055922 |
|electronics.audio.headphone     |1018478 |
|appliances.kitchen.refrigerators|863358  |
|appliances.kitchen.washer       |831234  |
|appliances.environment.vacuum   |772001  |
|apparel.shoes                   |759637  |
+--------------------------------+--------+
only showing top 10 rows

Top 10 categorías con más compras:
+--------------------------------+------+
|category_code_clean             |count |
+--------------------------------+------+
|electronics.smartphone          |337979|
|unknown                         |173411|
|electronics.audio.headphone     |30501 |
|electronics.video.tv            |21561 |

**Lealtad a la marca**

In [ ]:
from pyspark.sql.functions import countDistinct

# Usuarios que compran de una sola marca vs varias
marcas_por_usuario = df_clean.filter(col("event_type") == "purchase").groupBy("user_id") \
    .agg(countDistinct("brand_clean").alias("num_marcas_compradas")).cache()

print("Lealtad a la marca:")
marcas_por_usuario.describe("num_marcas_compradas").show()

lealtad = marcas_por_usuario.agg(
    spark_sum(when(col("num_marcas_compradas") == 1, 1).otherwise(0)).alias("leales"),
    spark_sum(when(col("num_marcas_compradas") > 1, 1).otherwise(0)).alias("diversos")
).collect()[0]

total = lealtad["leales"] + lealtad["diversos"]
print(f"Una sola marca: {lealtad['leales']:,} ({lealtad['leales']/total*100:.2f}%)")
print(f"Varias marcas: {lealtad['diversos']:,} ({lealtad['diversos']/total*100:.2f}%)")

Lealtad a la marca
+-------+--------------------+
|summary|num_marcas_compradas|
+-------+--------------------+
|  count|              347118|
|   mean|  1.3341054050783883|
| stddev|  0.8559955198932354|
|    min|                   1|
|    max|                  30|
+-------+--------------------+

Compradores de una sola marca: 273,457 (78.78%)
Compradores de varias marcas: 73,661 (21.22%)


# 2. Análisis de Precios y Valor

## 2.1 Distribución de Precios

In [60]:
# Estadísticas descriptivas de precios
from pyspark.sql.functions import expr, percentile_approx

print("Estadísticas globales de precios:")
df_clean.select("price").describe().show()

# Percentiles para detectar outliers
df_clean.select(
    percentile_approx("price", 0.25).alias("Q1"),
    percentile_approx("price", 0.50).alias("mediana"),
    percentile_approx("price", 0.75).alias("Q3"),
    percentile_approx("price", 0.95).alias("P95"),
    percentile_approx("price", 0.99).alias("P99")
).show()

Estadísticas globales de precios:
+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          42418544|
|   mean| 290.3132680178888|
| stddev|358.29733367226834|
|    min|               0.0|
|    max|           2574.07|
+-------+------------------+

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          42418544|
|   mean| 290.3132680178888|
| stddev|358.29733367226834|
|    min|               0.0|
|    max|           2574.07|
+-------+------------------+

+----+-------+------+-------+-------+
|  Q1|mediana|    Q3|    P95|    P99|
+----+-------+------+-------+-------+
|65.9| 162.68|358.57|1010.05|1741.33|
+----+-------+------+-------+-------+

+----+-------+------+-------+-------+
|  Q1|mediana|    Q3|    P95|    P99|
+----+-------+------+-------+-------+
|65.9| 162.68|358.57|1010.05|1741.33|
+----+-------+------+-------+-------+



__Segmentación de precios por cuantiles dentro de cada categoría__

In [ ]:
# Segmentación de precios por cuantiles dentro de cada categoría
from pyspark.sql.functions import percentile_approx

percentiles_por_categoria = df_clean.groupBy("category_code_clean").agg(
    percentile_approx("price", 0.33).alias("p33"),
    percentile_approx("price", 0.66).alias("p66")
).cache()

df_segmentado = df_clean.join(percentiles_por_categoria, "category_code_clean", "left").withColumn(
    "segmento_precio",
    when(col("price") < col("p33"), "barato")
    .when(col("price") < col("p66"), "medio")
    .otherwise("caro")
).cache()

print("Distribución de segmentos de precio:")
df_segmentado.groupBy("segmento_precio").count().orderBy("segmento_precio").show()

Distribución de segmentos de precio:
+---------------+--------+
|segmento_precio|   count|
+---------------+--------+
|         barato|13857551|
|           caro|14551579|
|          medio|14009414|
+---------------+--------+

+---------------+--------+
|segmento_precio|   count|
+---------------+--------+
|         barato|13857551|
|           caro|14551579|
|          medio|14009414|
+---------------+--------+



In [62]:
# Ejemplo: Ver umbrales por categoría
print("Umbrales de precio por categoría (muestra):")
percentiles_por_categoria.orderBy(col("p66").desc()).show(15, truncate=False)

Umbrales de precio por categoría (muestra):
+--------------------------------+------+------+
|category_code_clean             |p33   |p66   |
+--------------------------------+------+------+
|electronics.camera.photo        |442.71|900.41|
|computers.notebook              |420.6 |720.71|
|electronics.video.projector     |402.1 |705.72|
|computers.desktop               |173.75|617.75|
|furniture.living_room.sofa      |437.33|592.01|
|auto.accessories.winch          |344.77|573.14|
|electronics.smartphone          |215.57|485.54|
|electronics.video.tv            |269.2 |462.05|
|appliances.kitchen.dishwasher   |324.04|445.29|
|electronics.camera.video        |179.93|437.38|
|sport.trainer                   |200.78|383.53|
|kids.skates                     |257.15|368.81|
|appliances.kitchen.washer       |231.64|360.34|
|appliances.kitchen.refrigerators|242.63|360.32|
|electronics.tablet              |115.81|360.09|
+--------------------------------+------+------+
only showing top 15 rows


In [77]:
# Ejecutar si no se ejecutaron las bibliotecas antes
from pyspark.sql.functions import sum as spark_sum, when, col, count

In [68]:
# Conversión por segmento de precio
conversion_por_segmento = df_segmentado.groupBy("segmento_precio").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("views"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
).withColumn(
    "tasa_conversion", col("purchases") / col("views") * 100
)
print("Tasa de conversión por segmento de precio:")
conversion_por_segmento.orderBy("segmento_precio").show()

Tasa de conversión por segmento de precio:
+---------------+--------+---------+------------------+
|segmento_precio|   views|purchases|   tasa_conversion|
+---------------+--------+---------+------------------+
|         barato|13250908|   268130|2.0234839755886918|
|           caro|14059397|   230513|1.6395653383996485|
|          medio|13467023|   244130|1.8127985672854348|
+---------------+--------+---------+------------------+

+---------------+--------+---------+------------------+
|segmento_precio|   views|purchases|   tasa_conversion|
+---------------+--------+---------+------------------+
|         barato|13250908|   268130|2.0234839755886918|
|           caro|14059397|   230513|1.6395653383996485|
|          medio|13467023|   244130|1.8127985672854348|
+---------------+--------+---------+------------------+



__Precios por categoría__

In [78]:
# Ejecutar si no se ejecutaron las bibliotecas antes
from pyspark.sql.functions import avg, percentile_approx, min, max, split, col

In [72]:
# Estadísticas de precio por categoría principal
from pyspark.sql.functions import split

df_con_cat_principal = df_clean.withColumn(
    "categoria_principal", split(col("category_code_clean"), "\\.").getItem(0)
)

print("Precios por categoría principal:")
df_con_cat_principal.groupBy("categoria_principal").agg(
    avg("price").alias("precio_promedio"),
    percentile_approx("price", 0.5).alias("mediana"),
    min("price").alias("min"),
    max("price").alias("max")
).orderBy(col("precio_promedio").desc()).show(15)

Precios por categoría principal:
+-------------------+------------------+-------+---+-------+
|categoria_principal|   precio_promedio|mediana|min|    max|
+-------------------+------------------+-------+---+-------+
|          computers| 504.3028990900214| 360.34|0.0|2574.04|
|        electronics|412.26495465454116| 250.91|0.0|2574.07|
|              sport|  387.200470205765|  205.9|0.0|2573.81|
|       country_yard| 270.4435302890421| 238.86|0.0|2426.32|
|          furniture| 266.1417113404556| 176.53|0.0|2574.04|
|         appliances|223.47014463231133| 151.84|0.0|2574.04|
|            unknown|184.92899572328912|  82.34|0.0|2574.04|
|       construction| 164.2868837074333|  90.07|0.0|2571.19|
|               kids|160.57896687871408|  79.67|0.0|2571.49|
|               auto|140.92986071086915| 121.24|0.0|2165.76|
|            apparel| 81.59048442511641|  73.36|0.0| 913.79|
|        accessories| 60.90159841157904|  43.76|0.0|1717.16|
|           medicine| 50.95741519124206|  42.68|0.0|

__Variación de precios en el tiempo (detección de promociones)__

In [79]:
# Ejecutar si faltan bibliotecas
from pyspark.sql.functions import to_date, avg, percentile_approx, expr, col

In [ ]:
# Precio promedio por día (detectar promociones)
precio_por_dia = df_clean.withColumn("fecha", to_date(col("event_time"))).groupBy("fecha").agg(
    avg("price").alias("precio_promedio"),
    percentile_approx("price", 0.5).alias("mediana_precio")
).orderBy("fecha")

print("Variación de precio promedio por día:")
precio_por_dia.show(31)

# Detectar días con precios anómalos
stats_precio = precio_por_dia.agg(
    avg("precio_promedio").alias("media"),
    expr("stddev(precio_promedio)").alias("std")
).collect()[0]

dias_anomalos = precio_por_dia.filter(
    (col("precio_promedio") < stats_precio["media"] - 2 * stats_precio["std"]) |
    (col("precio_promedio") > stats_precio["media"] + 2 * stats_precio["std"])
)
print("Días con precios anómalos (±2):")
dias_anomalos.show()

Variación de precio promedio por día:
+----------+------------------+--------------+
|     fecha|   precio_promedio|mediana_precio|
+----------+------------------+--------------+
|2019-10-01|297.88166849318367|         161.9|
|2019-10-02|300.13027390450395|        164.17|
|2019-10-03|301.16020407873447|        168.84|
|2019-10-04|298.95175895995686|        169.86|
|2019-10-05|297.39146986385776|        167.29|
|2019-10-06| 301.1398038892656|        168.83|
|2019-10-07|296.06396959434045|        160.81|
|2019-10-08|277.88350016823733|        148.65|
|2019-10-09|281.78438398614804|        151.36|
|2019-10-10|289.72040033349964|         159.3|
|2019-10-11|282.06842161501646|        153.67|
|2019-10-12|281.92445732427416|        153.93|
|2019-10-13|279.54829235971624|        153.84|
|2019-10-14|298.04799310560736|        172.19|
|2019-10-15| 296.0082195538297|        170.92|
|2019-10-16|291.07615194758506|        167.31|
|2019-10-17|292.10875402958476|        166.23|
|2019-10-18| 287.70531

## 2.2 Análisis de Revenue

In [ ]:
# Revenue con cache para reutilizar
compras = df_clean.filter(col("event_type") == "purchase").cache()
revenue_total = compras.agg(spark_sum("price")).collect()[0][0]
print(f"Revenue total: ${revenue_total:,.2f}")

revenue_por_dia = compras.withColumn("fecha", to_date(col("event_time"))).groupBy("fecha").agg(
    spark_sum("price").alias("revenue"),
    count("*").alias("transacciones")
).orderBy("fecha")

print("\nRevenue por día:")
revenue_por_dia.show(31)
revenue_por_dia.describe("revenue").show()

Revenue total: $229,933,212.63

Revenue por día:

Revenue por día:
+----------+------------------+-------------+
|     fecha|           revenue|transacciones|
+----------+------------------+-------------+
|2019-10-01|6275579.0600000005|        19305|
|2019-10-02|        6213628.53|        19469|
|2019-10-03|6233782.9799999995|        19255|
|2019-10-04| 8623058.189999998|        27039|
|2019-10-05| 7341094.459999996|        23492|
|2019-10-06| 6737258.170000001|        22169|
|2019-10-07| 6348189.059999999|        21378|
|2019-10-08|        6819701.26|        23071|
|2019-10-09|        6855326.05|        22747|
|2019-10-10| 6665413.209999999|        21992|
|2019-10-11|        7716208.55|        26224|
|2019-10-12| 7307691.569999997|        25373|
|2019-10-13|        8457606.49|        29561|
|2019-10-14| 9356691.649999999|        28405|
|2019-10-15| 8652872.260000002|        26371|
|2019-10-16| 9747164.719999991|        31393|
|2019-10-17| 9026019.069999998|        28317|
|2019-10-18| 

__Revenue por categoría y marca__

In [81]:
# Revenue por categoría y marca (reutiliza compras cacheado)
revenue_categoria = compras.groupBy("category_code_clean").agg(
    spark_sum("price").alias("revenue"), count("*").alias("transacciones")
).withColumn("contribucion_pct", col("revenue") / revenue_total * 100).orderBy(col("revenue").desc())

print("Top 15 categorías por revenue:")
revenue_categoria.show(15, truncate=False)

revenue_marca = compras.groupBy("brand_clean").agg(
    spark_sum("price").alias("revenue"), count("*").alias("transacciones")
).withColumn("contribucion_pct", col("revenue") / revenue_total * 100).orderBy(col("revenue").desc())

print("Top 15 marcas por revenue:")
revenue_marca.show(15)

Top 15 categorías por revenue:
+--------------------------------+--------------------+-------------+-------------------+
|category_code_clean             |revenue             |transacciones|contribucion_pct   |
+--------------------------------+--------------------+-------------+-------------------+
|electronics.smartphone          |1.5703391396999988E8|337979       |68.29544639238053  |
|unknown                         |2.292216508000001E7 |173411       |9.96905354290227   |
|computers.notebook              |8978883.419999996   |15588        |3.9049962888347443 |
|electronics.video.tv            |8422119.379999999   |21561        |3.6628546540392866 |
|electronics.clocks              |4817089.040000001   |17903        |2.094994883471439  |
|appliances.kitchen.washer       |4658223.46          |16146        |2.025902829225391  |
|appliances.kitchen.refrigerators|3830077.0099999984  |11218        |1.6657345697001238 |
|electronics.audio.headphone     |3538807.1700000004  |30501        |

__Distribución de revenue por usuario (identificar ballenas)__

In [82]:
# LTV por usuario (reutiliza compras cacheado)
ltv_usuario = compras.groupBy("user_id").agg(
    spark_sum("price").alias("ltv"), count("*").alias("num_compras")
).orderBy(col("ltv").desc()).cache()

print("Distribución de LTV:")
ltv_usuario.describe("ltv").show()
print("Percentiles de LTV:")
ltv_usuario.selectExpr(
    "percentile_approx(ltv, 0.50) as p50",
    "percentile_approx(ltv, 0.75) as p75",
    "percentile_approx(ltv, 0.90) as p90",
    "percentile_approx(ltv, 0.95) as p95",
    "percentile_approx(ltv, 0.99) as p99"
).show()
print("Top 20 ballenas:")
ltv_usuario.show(20)

Distribución de LTV:
+-------+------------------+
|summary|               ltv|
+-------+------------------+
|  count|            347118|
|   mean|  662.406480303528|
| stddev|2074.2141910618398|
|    min|              0.88|
|    max|265569.51999999984|
+-------+------------------+

Percentiles de LTV:
+-------+------------------+
|summary|               ltv|
+-------+------------------+
|  count|            347118|
|   mean|  662.406480303528|
| stddev|2074.2141910618398|
|    min|              0.88|
|    max|265569.51999999984|
+-------+------------------+

Percentiles de LTV:
+------+------+-------+-------+-------+
|   p50|   p75|    p90|    p95|    p99|
+------+------+-------+-------+-------+
|246.52|594.53|1418.05|2336.73|6667.38|
+------+------+-------+-------+-------+

Top 20 ballenas:
+---------+------------------+-----------+
|  user_id|               ltv|num_compras|
+---------+------------------+-----------+
|519267944|265569.51999999984|        183|
|513117637|244499.9999999

__Análisis de Pareto (regla 80/20)__

In [83]:
# Análisis de Pareto (reutiliza ltv_usuario cacheado)
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

total_compradores = ltv_usuario.count()
w = Window.orderBy(col("ltv").desc())

pareto = ltv_usuario.withColumn("rank", row_number().over(w)) \
    .withColumn("ltv_acum", spark_sum("ltv").over(Window.orderBy(col("ltv").desc()).rowsBetween(Window.unboundedPreceding, 0))) \
    .withColumn("pct_revenue", col("ltv_acum") / revenue_total * 100) \
    .withColumn("pct_usuarios", col("rank") / total_compradores * 100)

usuarios_80pct = pareto.filter(col("pct_revenue") >= 80).select("rank", "pct_usuarios").first()
print(f"Total compradores: {total_compradores:,}")
print(f"80% revenue generado por {usuarios_80pct['rank']:,} usuarios ({usuarios_80pct['pct_usuarios']:.2f}%)")

pareto.withColumn("segmento",
    when(col("pct_revenue") <= 50, "top_50pct")
    .when(col("pct_revenue") <= 80, "mid_30pct")
    .otherwise("bottom_20pct")
).groupBy("segmento").agg(
    count("*").alias("usuarios"), spark_sum("ltv").alias("revenue")
).withColumn("pct_usuarios", col("usuarios") / total_compradores * 100) \
 .withColumn("pct_revenue", col("revenue") / revenue_total * 100).show()

Total compradores: 347,118
80% revenue generado por 98,087 usuarios (28.26%)
+------------+--------+--------------------+-----------------+------------------+
|    segmento|usuarios|             revenue|     pct_usuarios|       pct_revenue|
+------------+--------+--------------------+-----------------+------------------+
|   top_50pct|   25406|1.1496487390000086E8|7.319124908532546| 49.99924655730277|
|   mid_30pct|   72680| 6.898132837000144E7|20.93812478753623|30.000593468418884|
|bottom_20pct|  249032|4.5987010360003255E7|71.74275030393122|20.000159974280827|
+------------+--------+--------------------+-----------------+------------------+

+------------+--------+--------------------+-----------------+------------------+
|    segmento|usuarios|             revenue|     pct_usuarios|       pct_revenue|
+------------+--------+--------------------+-----------------+------------------+
|   top_50pct|   25406|1.1496487390000086E8|7.319124908532546| 49.99924655730277|
|   mid_30pct|   726